训练resnet18

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from facenet_pytorch import MTCNN
import torch.optim.lr_scheduler as lr_scheduler
import matplotlib.pyplot as plt

# 初始化列表来保存损失值和 MAE 数据
train_losses = []
valid_losses = []
train_maes = []
valid_maes = []

# 初始化 MTCNN 检测器
mtcnn = MTCNN()
target_size = (240, 320)

# 检查是否有可用的 GPU
if torch.cuda.is_available():
    available_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {available_gpus}")
    device = torch.device("cuda:0")
    gpu_name = torch.cuda.get_device_name(device)
    print(f"Using GPU: {gpu_name}")
else:
    device = torch.device("cpu")

def detect_and_crop_face(image, target_size=(240, 320)):
    # 使用 MTCNN 进行人脸检测
    boxes, _ = mtcnn.detect(image)

    if boxes is not None and len(boxes) > 0:
        # 获取第一个人脸的坐标
        x1, y1, x2, y2 = boxes[0]

        # 计算扩展边界的偏移量，扩展20%
        width = x2 - x1
        height = y2 - y1
        offset_x = width * 0.2  # 向外扩展20%
        offset_y = height * 0.2  # 向外扩展20%

        # 扩展人脸区域
        x1 = max(x1 - offset_x, 0)  # 确保坐标不小于0
        y1 = max(y1 - offset_y, 0)  # 确保坐标不小于0
        x2 = x2 + offset_x
        y2 = y2 + offset_y
        
        # 裁剪出人脸区域
        face = image.crop((x1, y1, x2, y2))  # 裁剪人脸区域

        # 计算保持比例的缩放
        scale = min(target_size[0] / face.width, target_size[1] / face.height)
        new_size = (int(face.width * scale), int(face.height * scale))
        face = face.resize(new_size, Image.Resampling.LANCZOS)

        # 如果人脸太小，使用 padding 保持目标尺寸
        left = (face.width - target_size[0]) / 2 if face.width > target_size[0] else 0
        top = (face.height - target_size[1]) / 2 if face.height > target_size[1] else 0
        right = (face.width + target_size[0]) / 2 if face.width > target_size[0] else face.width
        bottom = (face.height + target_size[1]) / 2 if face.height > target_size[1] else face.height

        face = face.crop((left, top, right, bottom))

        # 如果目标区域不足，进行填充
        face = face.resize(target_size, Image.Resampling.LANCZOS)
        plt.imshow(face)
        return face
    else:
        return Image.new('RGB', target_size, (0, 0, 0))  # 返回黑色图像作为替代


# 自定义数据集类
class BMI_Dataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.data_frame.iloc[idx, -1])
        image = Image.open(img_name).convert('RGB')
        image = detect_and_crop_face(image)
        bmi = self.data_frame.iloc[idx, -3]  # 获取BMI值
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(bmi, dtype=torch.float32)

# 数据集路径
csv_file = r'D:\A1-CV-DATABASE\data\train.csv'
root_dir = r'D:\A1-CV-DATABASE\data\face'

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 实例化数据集和数据加载器
dataset = BMI_Dataset(csv_file=csv_file, root_dir=root_dir, transform=transform)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

valid_dataset = BMI_Dataset(csv_file=csv_file, root_dir=root_dir, transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

# 使用预训练的 ResNet
class BMI_ResNet(nn.Module):
    def __init__(self):
        super(BMI_ResNet, self).__init__()
        self.resnet = models.resnet18(pretrained=True)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 1)

    def forward(self, x):
        return self.resnet(x)

model = BMI_ResNet().to(device)

# 损失函数和优化器
criterion = nn.SmoothL1Loss()
optimizer = AdamW(model.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 20

# 训练过程
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_mae = 0.0

    for images, bmis in tqdm(data_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        images = images.to(device)
        bmis = bmis.to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze()

        loss = criterion(outputs, bmis)
        mae = torch.mean(torch.abs(outputs - bmis))

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_mae += mae.item()

    avg_train_loss = running_loss / len(data_loader)
    avg_train_mae = running_mae / len(data_loader)
    train_losses.append(avg_train_loss)
    train_maes.append(avg_train_mae)

    model.eval()
    valid_loss = 0.0
    valid_mae = 0.0
    with torch.no_grad():
        for images, bmis in tqdm(valid_loader, desc=f'Validation Epoch {epoch+1}/{num_epochs}'):
            images = images.to(device)
            bmis = bmis.to(device)

            outputs = model(images).squeeze()

            loss = criterion(outputs, bmis)
            mae = torch.mean(torch.abs(outputs - bmis))

            valid_loss += loss.item()
            valid_mae += mae.item()

    avg_valid_loss = valid_loss / len(valid_loader)
    avg_valid_mae = valid_mae / len(valid_loader)
    valid_losses.append(avg_valid_loss)
    valid_maes.append(avg_valid_mae)

    scheduler.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_valid_loss:.4f}, '
          f'Train MAE: {avg_train_mae:.4f}, Validation MAE: {avg_valid_mae:.4f}')

# 绘制训练和验证 MAE 图表
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_maes, label='Train MAE', marker='o')
plt.plot(range(1, num_epochs + 1), valid_maes, label='Validation MAE', marker='o')
plt.title('Mean Absolute Error Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.grid(True)
plt.savefig(r'D:\A1-CV-DATABASE\1211-2-18.png')
plt.show()

# 保存模型
model_save_path = r'D:\A1-CV-DATABASE\model1211-2-18.pth'
torch.save(model.state_dict(), model_save_path)

程序输出结果：

训练resnet34

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from facenet_pytorch import MTCNN
import torch.optim.lr_scheduler as lr_scheduler
import matplotlib.pyplot as plt

# 初始化列表来保存损失值和 MAE 数据
train_losses = []
valid_losses = []
train_maes = []
valid_maes = []

# 初始化 MTCNN 检测器
mtcnn = MTCNN()
target_size = (240, 320)

# 检查是否有可用的 GPU
if torch.cuda.is_available():
    available_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {available_gpus}")
    device = torch.device("cuda:0")
    gpu_name = torch.cuda.get_device_name(device)
    print(f"Using GPU: {gpu_name}")
else:
    device = torch.device("cpu")

def detect_and_crop_face(image, target_size=(240, 320)):
    # 使用 MTCNN 进行人脸检测
    boxes, _ = mtcnn.detect(image)

    if boxes is not None and len(boxes) > 0:
        # 获取第一个人脸的坐标
        x1, y1, x2, y2 = boxes[0]

        # 计算扩展边界的偏移量，扩展20%
        width = x2 - x1
        height = y2 - y1
        offset_x = width * 0.2  # 向外扩展20%
        offset_y = height * 0.2  # 向外扩展20%

        # 扩展人脸区域
        x1 = max(x1 - offset_x, 0)  # 确保坐标不小于0
        y1 = max(y1 - offset_y, 0)  # 确保坐标不小于0
        x2 = x2 + offset_x
        y2 = y2 + offset_y
        
        # 裁剪出人脸区域
        face = image.crop((x1, y1, x2, y2))  # 裁剪人脸区域

        # 计算保持比例的缩放
        scale = min(target_size[0] / face.width, target_size[1] / face.height)
        new_size = (int(face.width * scale), int(face.height * scale))
        face = face.resize(new_size, Image.Resampling.LANCZOS)

        # 如果人脸太小，使用 padding 保持目标尺寸
        left = (face.width - target_size[0]) / 2 if face.width > target_size[0] else 0
        top = (face.height - target_size[1]) / 2 if face.height > target_size[1] else 0
        right = (face.width + target_size[0]) / 2 if face.width > target_size[0] else face.width
        bottom = (face.height + target_size[1]) / 2 if face.height > target_size[1] else face.height

        face = face.crop((left, top, right, bottom))

        # 如果目标区域不足，进行填充
        face = face.resize(target_size, Image.Resampling.LANCZOS)
        plt.imshow(face)
        return face
    else:
        return Image.new('RGB', target_size, (0, 0, 0))  # 返回黑色图像作为替代


# 自定义数据集类
class BMI_Dataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.data_frame.iloc[idx, -1])
        image = Image.open(img_name).convert('RGB')
        image = detect_and_crop_face(image)
        bmi = self.data_frame.iloc[idx, -3]  # 获取BMI值
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(bmi, dtype=torch.float32)

# 数据集路径
csv_file = r'D:\A1-CV-DATABASE\data\train.csv'
root_dir = r'D:\A1-CV-DATABASE\data\face'

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 实例化数据集和数据加载器
dataset = BMI_Dataset(csv_file=csv_file, root_dir=root_dir, transform=transform)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

valid_dataset = BMI_Dataset(csv_file=csv_file, root_dir=root_dir, transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

# 使用预训练的 ResNet
class BMI_ResNet(nn.Module):
    def __init__(self):
        super(BMI_ResNet, self).__init__()
        self.resnet = models.resnet34(pretrained=True)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 1)

    def forward(self, x):
        return self.resnet(x)

model = BMI_ResNet().to(device)

# 损失函数和优化器
criterion = nn.SmoothL1Loss()
optimizer = AdamW(model.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 20

# 训练过程
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_mae = 0.0

    for images, bmis in tqdm(data_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        images = images.to(device)
        bmis = bmis.to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze()

        loss = criterion(outputs, bmis)
        mae = torch.mean(torch.abs(outputs - bmis))

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_mae += mae.item()

    avg_train_loss = running_loss / len(data_loader)
    avg_train_mae = running_mae / len(data_loader)
    train_losses.append(avg_train_loss)
    train_maes.append(avg_train_mae)

    model.eval()
    valid_loss = 0.0
    valid_mae = 0.0
    with torch.no_grad():
        for images, bmis in tqdm(valid_loader, desc=f'Validation Epoch {epoch+1}/{num_epochs}'):
            images = images.to(device)
            bmis = bmis.to(device)

            outputs = model(images).squeeze()

            loss = criterion(outputs, bmis)
            mae = torch.mean(torch.abs(outputs - bmis))

            valid_loss += loss.item()
            valid_mae += mae.item()

    avg_valid_loss = valid_loss / len(valid_loader)
    avg_valid_mae = valid_mae / len(valid_loader)
    valid_losses.append(avg_valid_loss)
    valid_maes.append(avg_valid_mae)

    scheduler.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_valid_loss:.4f}, '
          f'Train MAE: {avg_train_mae:.4f}, Validation MAE: {avg_valid_mae:.4f}')

# 绘制训练和验证 MAE 图表
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_maes, label='Train MAE', marker='o')
plt.plot(range(1, num_epochs + 1), valid_maes, label='Validation MAE', marker='o')
plt.title('Mean Absolute Error Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.grid(True)
plt.savefig(r'D:\A1-CV-DATABASE\1211-2-34.png')
plt.show()

# 保存模型
model_save_path = r'D:\A1-CV-DATABASE\model1211-2-34.pth'
torch.save(model.state_dict(), model_save_path)

程序输出结果：

训练resnet50

In [ ]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms, models
import torch.nn as nn
from torch.optim import AdamW
from tqdm import tqdm
from facenet_pytorch import MTCNN
import torch.optim.lr_scheduler as lr_scheduler
import matplotlib.pyplot as plt

# 初始化列表来保存损失值和 MAE 数据
train_losses = []
valid_losses = []
train_maes = []
valid_maes = []

# 初始化 MTCNN 检测器
mtcnn = MTCNN()
target_size = (240, 320)

# 检查是否有可用的 GPU
if torch.cuda.is_available():
    available_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {available_gpus}")
    device = torch.device("cuda:0")
    gpu_name = torch.cuda.get_device_name(device)
    print(f"Using GPU: {gpu_name}")
else:
    device = torch.device("cpu")

def detect_and_crop_face(image, target_size=(240, 320)):
    # 使用 MTCNN 进行人脸检测
    boxes, _ = mtcnn.detect(image)

    if boxes is not None and len(boxes) > 0:
        # 获取第一个人脸的坐标
        x1, y1, x2, y2 = boxes[0]

        # 计算扩展边界的偏移量，扩展20%
        width = x2 - x1
        height = y2 - y1
        offset_x = width * 0.2  # 向外扩展20%
        offset_y = height * 0.2  # 向外扩展20%

        # 扩展人脸区域
        x1 = max(x1 - offset_x, 0)  # 确保坐标不小于0
        y1 = max(y1 - offset_y, 0)  # 确保坐标不小于0
        x2 = x2 + offset_x
        y2 = y2 + offset_y
        
        # 裁剪出人脸区域
        face = image.crop((x1, y1, x2, y2))  # 裁剪人脸区域

        # 计算保持比例的缩放
        scale = min(target_size[0] / face.width, target_size[1] / face.height)
        new_size = (int(face.width * scale), int(face.height * scale))
        face = face.resize(new_size, Image.Resampling.LANCZOS)

        # 如果人脸太小，使用 padding 保持目标尺寸
        left = (face.width - target_size[0]) / 2 if face.width > target_size[0] else 0
        top = (face.height - target_size[1]) / 2 if face.height > target_size[1] else 0
        right = (face.width + target_size[0]) / 2 if face.width > target_size[0] else face.width
        bottom = (face.height + target_size[1]) / 2 if face.height > target_size[1] else face.height

        face = face.crop((left, top, right, bottom))

        # 如果目标区域不足，进行填充
        face = face.resize(target_size, Image.Resampling.LANCZOS)
        plt.imshow(face)
        return face
    else:
        return Image.new('RGB', target_size, (0, 0, 0))  # 返回黑色图像作为替代


# 自定义数据集类
class BMI_Dataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data_frame = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform = transform

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.data_frame.iloc[idx, -1])
        image = Image.open(img_name).convert('RGB')
        image = detect_and_crop_face(image)
        bmi = self.data_frame.iloc[idx, -3]  # 获取BMI值
        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(bmi, dtype=torch.float32)

# 数据集路径
csv_file = r'D:\A1-CV-DATABASE\data\train.csv'
root_dir = r'D:\A1-CV-DATABASE\data\face'

transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 实例化数据集和数据加载器
dataset = BMI_Dataset(csv_file=csv_file, root_dir=root_dir, transform=transform)
data_loader = DataLoader(dataset, batch_size=32, shuffle=True)

valid_dataset = BMI_Dataset(csv_file=csv_file, root_dir=root_dir, transform=transform)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False)

# 使用预训练的 ResNet
class BMI_ResNet(nn.Module):
    def __init__(self):
        super(BMI_ResNet, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 1)

    def forward(self, x):
        return self.resnet(x)

model = BMI_ResNet().to(device)

# 损失函数和优化器
criterion = nn.SmoothL1Loss()
optimizer = AdamW(model.parameters(), lr=0.001)
scheduler = lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 20

# 训练过程
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    running_mae = 0.0

    for images, bmis in tqdm(data_loader, desc=f'Epoch {epoch+1}/{num_epochs}'):
        images = images.to(device)
        bmis = bmis.to(device)

        optimizer.zero_grad()
        outputs = model(images).squeeze()

        loss = criterion(outputs, bmis)
        mae = torch.mean(torch.abs(outputs - bmis))

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        running_mae += mae.item()

    avg_train_loss = running_loss / len(data_loader)
    avg_train_mae = running_mae / len(data_loader)
    train_losses.append(avg_train_loss)
    train_maes.append(avg_train_mae)

    model.eval()
    valid_loss = 0.0
    valid_mae = 0.0
    with torch.no_grad():
        for images, bmis in tqdm(valid_loader, desc=f'Validation Epoch {epoch+1}/{num_epochs}'):
            images = images.to(device)
            bmis = bmis.to(device)

            outputs = model(images).squeeze()

            loss = criterion(outputs, bmis)
            mae = torch.mean(torch.abs(outputs - bmis))

            valid_loss += loss.item()
            valid_mae += mae.item()

    avg_valid_loss = valid_loss / len(valid_loader)
    avg_valid_mae = valid_mae / len(valid_loader)
    valid_losses.append(avg_valid_loss)
    valid_maes.append(avg_valid_mae)

    scheduler.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_train_loss:.4f}, Validation Loss: {avg_valid_loss:.4f}, '
          f'Train MAE: {avg_train_mae:.4f}, Validation MAE: {avg_valid_mae:.4f}')

# 绘制训练和验证 MAE 图表
plt.figure(figsize=(10, 6))
plt.plot(range(1, num_epochs + 1), train_maes, label='Train MAE', marker='o')
plt.plot(range(1, num_epochs + 1), valid_maes, label='Validation MAE', marker='o')
plt.title('Mean Absolute Error Over Epochs')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.grid(True)
plt.savefig(r'D:\A1-CV-DATABASE\1211-2-50.png')
plt.show()

# 保存模型
model_save_path = r'D:\A1-CV-DATABASE\model1211-2-50.pth'
torch.save(model.state_dict(), model_save_path)

程序输出结果：

模型测试代码：

In [ ]:
import torch
from PIL import Image
from torchvision import transforms,models
import torch.nn as nn
from facenet_pytorch import MTCNN
import matplotlib.pyplot as plt
from PIL import ImageDraw

# 定义加载模型的路径
model_save_path = r'D:\A1-CV-DATABASE\testcv_resnet124-2.pth'
# 检查是否有可用的 GPU
if torch.cuda.is_available():
    available_gpus = torch.cuda.device_count()
    print(f"Available GPUs: {available_gpus}")
    device = torch.device("cuda:0")
    gpu_name = torch.cuda.get_device_name(device)
    print(f"Using GPU: {gpu_name}")
else:
    device = torch.device("cpu")
    
mtcnn = MTCNN()

# 加载预训练的 ResNet
class BMI_ResNet(nn.Module):
    def __init__(self):
        super(BMI_ResNet, self).__init__()
        self.resnet = models.resnet18(pretrained=True)
        # 修改最后的全连接层
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, 1)

    def forward(self, x):
        return self.resnet(x)
        
# 加载预训练模型
model = BMI_ResNet().to(device)  # 请确保model定义和之前训练时的模型一致
model.load_state_dict(torch.load(model_save_path))
model.eval()  # 设置为评估模式

transform = transforms.Compose([
    transforms.Resize((224, 224)),  
    transforms.ToTensor(),  # 将图像转为张量
    
])

def detect_and_crop_face(image, target_size=(240,320)):
    # 使用 MTCNN 进行人脸检测
    boxes, _ = mtcnn.detect(image)
    if boxes is not None and len(boxes) > 0:
        # 获取第一个人脸的坐标
        x1, y1, x2, y2 = boxes[0]
        

        print(boxes)
        draw = ImageDraw.Draw(image)
        # 绘制矩形框
        for box in boxes:
            # 将box中的坐标转换为整数
            box = [int(coord) for coord in box]
            draw.rectangle(box, outline=(255, 0, 0), width=1)

            # 显示图像
        plt.imshow(image)
        plt.axis('off')  # 不显示坐标轴
        plt.show()
        chang=x2-x1
        kuan=y2-y1
        print('横轴长：{:.2f}, 竖轴宽：{:.2f}'.format(chang, kuan))
        
        # 裁剪出人脸区域
        face = image.crop((x1, y1, x2, y2))  # 裁剪人脸区域

        # 计算保持比例的缩放
        scale = min(target_size[0] / face.width, target_size[1] / face.height)
        new_size = (int(face.width * scale), int(face.height * scale))
        face = face.resize(new_size, Image.Resampling.LANCZOS)

        # 如果人脸太小，使用 padding 保持目标尺寸
        left = (face.width - target_size[0]) / 2 if face.width > target_size[0] else 0
        top = (face.height - target_size[1]) / 2 if face.height > target_size[1] else 0
        right = (face.width + target_size[0]) / 2 if face.width > target_size[0] else face.width
        bottom = (face.height + target_size[1]) / 2 if face.height > target_size[1] else face.height

        face = face.crop((left, top, right, bottom))

        # 如果目标区域不足，进行填充
        face = face.resize(target_size, Image.Resampling.LANCZOS)
        plt.imshow(face)
        return face
    else:
        print("No face detected, returning a blank image.")
        return Image.new('RGB', target_size, (0, 0, 0))  # 返回黑色图像作为替代

# 测试单张图片
def test_single_image(image_path):
    # 打开图片
    image = Image.open(image_path).convert('RGB')
    
    # 应用预处理
    image = detect_and_crop_face(image)
    image = transform(image)
    # 将图像转移到 GPU（如果有）
    image = image.unsqueeze(0).to(device)  # 增加 batch 维度并移至 GPU
    
    # 进行预测
    with torch.no_grad():  # 不需要计算梯度
        output = model(image)
    
    # 输出预测的BMI
    predicted_bmi = output.item()  # 获取单个值
    print(f"Predicted BMI: {predicted_bmi:.4f}")

# 测试图片路径
image_path = r'D:\A1-CV-DATABASE\data\test\single_face\3.jpg'

# 调用测试函数
test_single_image(image_path)